# Mini-projet — Agent MCP multi-outils avec Gemini et LangGraph

Ce notebook :
1. installe les dépendances,
2. démarre deux serveurs MCP tiers (`filesystem`, `git`),
3. **crée et démarre un serveur MCP personnalisé** avec `FastMCP` (outils "notes"),
4. connecte un LLM **Gemini** à l'ensemble des outils,
5. **orchestre ces outils via un agent LangGraph** (ReAct) capable de décider lui-même quel outil appeler pour répondre à une requête.

### 1. Installation des dépendances Python
On ajoute `mcp` explicitement (fournit `mcp.server.fastmcp.FastMCP`, utilisé à l'étape 3 pour le serveur personnalisé) ainsi que `langgraph` (déjà présent) pour l'orchestration.

In [ ]:
%pip install -qU \
  "langchain>=0.3" \
  "langgraph>=0.2" \
  "langchain-google-genai>=2.0" \
  "google-genai>=1.0" \
  "langchain-mcp-adapters==0.2.1" \
  "mcp" \
  "nest_asyncio" \
  "mcp-server-git"


### 2. Installation de Node.js / npx
Nécessaire pour lancer le serveur MCP `@modelcontextprotocol/server-filesystem` via `npx`.

In [ ]:
!apt-get -qq update
!apt-get -qq install -y nodejs npm
!node --version
!npx --version


### 3. Création d'un serveur MCP personnalisé avec FastMCP

Le projet demandait explicitement de démontrer la capacité à créer et intégrer un **outil MCP maison**. On écrit ici un petit serveur `notes-server`, avec deux outils :
- `add_note(text)` : ajoute une note en mémoire,
- `list_notes()` : liste les notes déjà enregistrées.

`FastMCP` génère automatiquement le schéma des outils à partir des annotations de type et des docstrings Python, ce qui rend l'écriture d'un serveur MCP personnalisé très rapide.

In [ ]:
import os

SERVER_DIR = "/content/mcp_servers"
os.makedirs(SERVER_DIR, exist_ok=True)
CUSTOM_SERVER_PATH = os.path.join(SERVER_DIR, "custom_notes_server.py")

custom_server_code = '''
from mcp.server.fastmcp import FastMCP

mcp = FastMCP("notes-server")

_notes = []


@mcp.tool()
def add_note(text: str) -> str:
    """Add a note to the in-memory notes list and return a confirmation."""
    _notes.append(text)
    return f"Note added. Total notes: {len(_notes)}"


@mcp.tool()
def list_notes() -> str:
    """Return all notes currently stored, one per line."""
    if not _notes:
        return "No notes yet."
    return "\n".join(f"{i + 1}. {n}" for i, n in enumerate(_notes))


if __name__ == "__main__":
    mcp.run()
'''

with open(CUSTOM_SERVER_PATH, "w") as f:
    f.write(custom_server_code)

print("Custom MCP server written to:", CUSTOM_SERVER_PATH)


### 4. Connect to MCP servers from your agent runtime

On enregistre les **trois** serveurs MCP (`filesystem`, `git`, et notre serveur `custom_notes` créé à l'étape 3) auprès d'un `MultiServerMCPClient`.

**Rappel sur l'erreur `fileno()` :** dans Colab/Jupyter, `sys.stderr` est remplacé par un objet `ipykernel.iostream.OutStream` qui n'implémente pas `fileno()`. Or `stdio_client` transmet `sys.stderr` directement à `subprocess.Popen` comme flux d'erreur du sous-processus, ce qui exige un vrai descripteur de fichier — d'où le `UnsupportedOperation: fileno`. Comme les appels d'outils MCP peuvent ouvrir une nouvelle session (et donc un nouveau sous-processus) à **chaque invocation**, et pas seulement lors de la récupération de la liste des outils, on encapsule **tout le pipeline** (connexion, construction de l'agent, exécution des requêtes) dans une seule redirection temporaire de `sys.stderr`, plutôt que de ne protéger que `get_tools()`.

In [ ]:
import asyncio
import subprocess
import sys

import nest_asyncio
from langchain_mcp_adapters.client import MultiServerMCPClient

nest_asyncio.apply()

WORKDIR = "/content"  # Define the working directory for the servers
os.makedirs(WORKDIR, exist_ok=True)

# Le serveur MCP "git" a besoin d'un dépôt git existant dans WORKDIR.
if not os.path.isdir(os.path.join(WORKDIR, ".git")):
    subprocess.run(["git", "init", WORKDIR], check=True)

mcp_connections = {
    "filesystem": {
        "transport": "stdio",
        "command": "npx",
        "args": ["-y", "@modelcontextprotocol/server-filesystem", WORKDIR],
    },
    "git": {
        "transport": "stdio",
        "command": sys.executable,  # garantit le bon interpréteur Python
        "args": ["-m", "mcp_server_git", "--repository", WORKDIR],
    },
    "custom_notes": {
        "transport": "stdio",
        "command": sys.executable,
        "args": [CUSTOM_SERVER_PATH],
    },
}

print("Serveurs MCP configurés :", list(mcp_connections.keys()))


### 5. Configuration du LLM Gemini

La clé d'API est lue depuis les "Secrets" de Colab (`GOOGLE_API_KEY`) si disponible, sinon elle est demandée de façon masquée. Adaptez `model=` si besoin au nom du modèle Gemini disponible sur votre compte.

In [ ]:
import os
from getpass import getpass

if not os.environ.get("GOOGLE_API_KEY"):
    try:
        from google.colab import userdata
        os.environ["GOOGLE_API_KEY"] = userdata.get("GOOGLE_API_KEY")
    except Exception:
        os.environ["GOOGLE_API_KEY"] = getpass("Entrez votre clé API Google (Gemini) : ")

from langchain_google_genai import ChatGoogleGenerativeAI

llm = ChatGoogleGenerativeAI(model="gemini-2.0-flash", temperature=0)
print("LLM Gemini prêt.")


### 6. Construction de l'agent LangGraph (ReAct) et exécution

C'est ici que se fait l'orchestration demandée : `create_react_agent` construit un agent LangGraph qui, à chaque tour, décide lui-même quel outil MCP appeler (filesystem, git ou notes) en fonction de la requête de l'utilisateur, exécute l'appel, observe le résultat, et recommence jusqu'à pouvoir répondre.

Toute la séquence (connexion aux 3 serveurs MCP, construction de l'agent, exécution de la requête de test) est encapsulée dans une seule coroutine `main()`, elle-même protégée par la redirection temporaire de `sys.stderr` décrite plus haut.

In [ ]:
from langgraph.prebuilt import create_react_agent


async def main(query: str):
    client = MultiServerMCPClient(mcp_connections, tool_name_prefix=True)
    tools = await client.get_tools()
    print("Tool count:", len(tools))
    print([t.name for t in tools])

    agent = create_react_agent(llm, tools)

    result = await agent.ainvoke({"messages": [("user", query)]})
    return result["messages"][-1].content


test_query = (
    "List the files in the working directory, then add a note saying "
    "'MCP demo complete', then show the current git status of the repository."
)

original_stderr = sys.stderr
try:
    sys.stderr = sys.__stderr__
    response = asyncio.get_event_loop().run_until_complete(main(test_query))
finally:
    sys.stderr = original_stderr

print("\n=== Réponse de l'agent ===")
print(response)
